# 02c — Tuneo de hiperparámetros

Rama de la **optimización**. La columna vertebral está en
`02_supervisado.ipynb`.

La respuesta corta: **el tuneo no mejora el resultado**. Con CatBoost y
`block_pca`, el modelo con valores fijos da AUC 0,940 y recall@5 0,645; el tuneado
por LOSO anidado da 0,938 y 0,630. La diferencia es menor que la variación entre
escenarios, y va en la dirección contraria a la esperada.

El recall interno que reporta la búsqueda (0,663) queda por encima del externo
(0,630): la validación interna sobreestima, el mismo efecto que ya se midió con
Random Forest. Conclusión: los valores fijos ya estaban en el techo de la grilla.

El código que generó el CSV está al final. **No hace falta re-ejecutarlo**: el
resultado se lee de `resultados/blockpca_tune_catboost.csv`.


## Antes de empezar: si lo abrís en Google Colab

Este notebook vive en el repo y lee `datasets/`, `src/`, `resultados/` y `modelos/` desde la raíz del proyecto. En Colab esos archivos no existen, así que la celda de abajo clona el repo en `/content/data4cyber` y se para ahí.

**Cómo abrirlo:**

1. En Colab: *Archivo → Abrir notebook → GitHub*, pegá la URL del repo y elegí el notebook.
2. Ejecutá la primera celda. Clona el repo (una sola vez por sesión) y queda listo para correr.

Fuera de Colab la celda no hace nada.


In [2]:
# --- Preparar el entorno: solo hace algo si estás en Google Colab ---
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/maxiricalde/data4cyber.git"
DESTINO = Path("/content/data4cyber")

try:
    import google.colab  # noqa: F401

    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    # CatBoost no viene preinstalado en Colab; lo instalamos una sola vez.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost"], check=False)
    # Una sola vez por sesión: clonar el repo y pararse en su raíz.
    if not (DESTINO / "src" / "data.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(DESTINO)], check=True)
    os.chdir(DESTINO)
    print("proyecto en:", os.getcwd())
else:
    print("corriendo local: no hay nada que preparar")


corriendo local: no hay nada que preparar


In [3]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# El notebook vive en notebooks/, pero los datasets y src/ están en la raíz del proyecto.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").exists())
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src.data import build_matrices  # solo se usa para quitar la capa de anotación
from src.features import detrend
from src.reduce import block_pca, corr_filter, dimensionless_columns, physical_features

pd.set_option("display.width", 140)
print("raíz del proyecto:", RAIZ)
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, StratifiedKFold


raíz del proyecto: /media/windows/Users/pablo/Documents/UNSA/4to/MetodosCuantitativos/TPFinal/tpfinal-colab


## 0. Los datos

Cada archivo CSV corresponde a un escenario. Los archivos comparten las mismas columnas: mediciones eléctricas, estado de la batería y variables del perfil del escenario (`Profile.*`). Más adelante veremos qué información aportan estas últimas y por qué hay que interpretarlas con cuidado.


In [4]:
ESCENARIOS = ("S0", "S1", "S1alt", "S2", "S3", "S4", "S5", "S6")

frames = pd.concat(
    [pd.read_csv(f"datasets/dataset{nombre}.csv").assign(scenario=nombre) for nombre in ESCENARIOS],
    ignore_index=True,
)

print("filas totales:", frames.shape)
print(frames["scenario"].value_counts().sort_index())

filas totales: (14354, 157)
scenario
S0       1798
S1       1798
S1alt    1792
S2       1768
S3       1800
S4       1798
S5       1800
S6       1800
Name: count, dtype: int64


`build_matrices` separa las columnas que usaremos para aprender de las anotaciones que sirven para describir los datos. Entre las que se apartan están `attack_active`, `attack_phase`, `timestamp` y el precio de MQTT; el precio se excluye porque puede revelar directamente la etiqueta del ataque. Además excluimos `Profile.*` — las siete columnas de contexto reproducido — por la fuga que mostramos recién arriba.

La función devuelve:

- `X`: las 132 variables de entrada (también llamadas *features*), ya sin `Profile.*`.
- `y_binary`: la respuesta correcta de cada fila: `1` significa ataque y `0`, benigno.
- `scenario`: el escenario al que pertenece cada fila.

In [5]:
# Sin Profile.*: la fuga de identidad de escenario que mostramos arriba.
m = build_matrices(frames, include_profile=False)

X_crudo = m.X
y = m.y_binary
escenario = m.scenario

print("X:", X_crudo.shape)
print(f"ataques: {int(y.sum()):,} ({100 * y.mean():.1f} %)")

X: (14354, 139)
ataques: 7,586 (52.8 %)


In [7]:
# Rango móvil de 120 s, causal, dentro de cada escenario. Es `detrend` de
# src/features.py con method="range": producción aplica exactamente esta
# transformación, así que el cuaderno evalúa lo mismo que se despliega.
X = detrend(X_crudo, escenario, method="range", window=120)
print("features:", X.shape[1], "| filas:", X.shape[0])


features: 139 | filas: 14354


## 1. El resultado, leído del CSV


In [ ]:
tuneo = pd.read_csv("resultados/blockpca_tune_catboost.csv").set_index("fold")
display(tuneo.rename(columns={
    "best_depth": "profundidad", "best_lr": "learning rate",
    "best_iterations": "iteraciones", "recall5_interno": "recall@5 interno",
    "auc_test": "AUC (test)", "recall5_test": "recall@5 (test)",
}).round(3))

sin_tunear = (
    pd.read_csv("resultados/blockpca_loso.csv")
    .query("model == 'catboost'").set_index("fold")
)
folds = ["S1", "S1alt", "S2", "S3", "S4", "S5", "S6"]
comparacion = pd.DataFrame({
    "sin tunear (AUC)": sin_tunear.loc[folds, "auc"],
    "tuneado (AUC)": tuneo.loc[folds, "auc_test"],
    "sin tunear (recall@5)": sin_tunear.loc[folds, "recall_5"],
    "tuneado (recall@5)": tuneo.loc[folds, "recall5_test"],
})
display(comparacion.round(3))
display(comparacion.mean().round(3).to_frame("media").T)



## 2. Por qué CatBoost y por qué una grilla chica

El código de referencia de la cursada usa `GridSearchCV` y `RandomizedSearchCV`. Acá seguimos esa línea y **no** usamos optimización bayesiana: con LOSO anidado (8 folds externos) y cuatro modelos, una búsqueda bayesiana dispara el cómputo en Colab sin una ganancia esperable — la sección 3.1 de `02_supervisado` ya mostró que el margen de los hiperparámetros (~0,01 de recall) es menor que la variación entre escenarios (~0,13 de AUC).

Tuneamos **CatBoost**, que es el modelo nuevo y el que tiene más probabilidad de estar lejos de su mejor configuración. La grilla es chica a propósito: profundidad y learning rate, con 300 iteraciones fijas. El `block_pca` se reajusta en cada split interno, así que la validación interna nunca lo vio.


In [ ]:
# NO hace falta re-ejecutar esta celda: el CSV ya está en resultados/.
# Se deja como código generador, para que el número sea reproducible.

parametros_catboost = [(4, 0.05), (4, 0.10), (6, 0.05), (6, 0.10)]
filas_tuneo = []

for retenido in ESCENARIOS:
    test = (escenario == retenido).to_numpy()
    train = ~test
    grupos_train = escenario.loc[train].to_numpy()

    # Validación interna: dejar un escenario con ataque afuera del entrenamiento externo.
    divisiones = []
    for i_train, i_valid in LeaveOneGroupOut().split(
        X.loc[train], y.loc[train], groups=grupos_train
    ):
        if len(np.unique(y.loc[train].iloc[i_valid])) == 2:
            divisiones.append((i_train, i_valid))

    mejor = None
    for profundidad, learning_rate in parametros_catboost:
        internos = []
        for i_train, i_valid in divisiones:
            X_tr, X_va = block_pca(X.loc[train].iloc[i_train], X.loc[train].iloc[i_valid])
            modelo = CatBoostClassifier(
                iterations=300,
                depth=profundidad,
                learning_rate=learning_rate,
                random_seed=42,
                verbose=0,
            )
            modelo.fit(
                X_tr,
                y.loc[train].iloc[i_train],
                sample_weight=pesos_balanceados(y.loc[train].iloc[i_train]),
            )
            score_interno = modelo.predict_proba(X_va)[:, 1]
            internos.append(recall_con_fpr(y.loc[train].iloc[i_valid], score_interno, 0.05))
        media = np.mean(internos)
        if mejor is None or media > mejor[1]:
            mejor = ((profundidad, learning_rate), media)

    (profundidad, learning_rate), interna = mejor
    X_tr, X_te = block_pca(X.loc[train], X.loc[test])
    modelo = CatBoostClassifier(
        iterations=300,
        depth=profundidad,
        learning_rate=learning_rate,
        random_seed=42,
        verbose=0,
    )
    modelo.fit(X_tr, y.loc[train], sample_weight=pesos_balanceados(y.loc[train]))
    score_test = modelo.predict_proba(X_te)[:, 1]
    y_test = y.loc[test].to_numpy()
    dos_clases = len(np.unique(y_test)) == 2

    filas_tuneo.append(
        {
            "fold": retenido,
            "profundidad": profundidad,
            "learning_rate": learning_rate,
            "recall@5% interno": round(interna, 3),
            "AUC test": roc_auc_score(y_test, score_test) if dos_clases else np.nan,
            "recall@5% test": recall_con_fpr(y_test, score_test, 0.05) if dos_clases else np.nan,
        }
    )

tabla_tuneo = pd.DataFrame(filas_tuneo)
display(tabla_tuneo.round(3))


### Cómo salió el tuneo

| | AUC medio | recall@5 |
|---|---|---|
| CatBoost con valores fijos | **0,940** | **0,645** |
| CatBoost tuneado (LOSO anidado) | 0,938 | 0,630 |

La diferencia es menor que la variación entre escenarios. La combinación ganadora
cambia de fold en fold (profundidad 4 en S1, S2, S3 y S5; 6 en S1alt, S4 y S6), y
el recall interno que reporta la búsqueda (0,663) queda por encima del que
después se obtiene en el escenario reservado (0,630): la validación interna
sobreestima, el mismo efecto que ya medimos con Random Forest.

La lectura: **la configuración pesa menos que el escenario**. Para estos datos y
esta reducción, los valores fijos ya estaban en el techo de la grilla. Es un
resultado válido y se informa como tal: el tuneo confirma que no hay una
combinación escondida que cambie el resultado.


## Cierre de la rama

El tuneo no cambió el fondo: el margen de los hiperparámetros es menor que la
variación entre escenarios. Se informa el resultado negativo tal como salió.

Volver a la columna vertebral: [`02_supervisado.ipynb`](02_supervisado.ipynb).
Otras ramas: [`02a_ventana.ipynb`](02a_ventana.ipynb),
[`02b_reduccion.ipynb`](02b_reduccion.ipynb).
